# Project Phase 2

## Short Introduction to our Dataset

### Original Research Question

*"How heavily did a state’s COVID-19 measurement policy have an impact on mental health
compared to its counterparts?"*

### Dataset Descriptions

- 1. CDC Depression Data *(data.cdc.gov)*: ...
- 2. Covid Policy Dataset *(github.com/OxCGRT)*: ...

### Joins Explained

...

### Final Dataset Shape (rows x columns)

- Columns: ...
- Rows: 1...


## Part 1 - Dataset Preparation and Joins

- What column you joined on
- What type of join you used (inner, left, etc.)
- Number of rows before the join
- Number of rows after the join
- Why the row count changed or stayed the same


### 1.1 Download the datasets

In [65]:
# Download the required datasets
import os
import urllib.request

os.makedirs("dataset", exist_ok=True)

file_path_1 = "dataset/cdc_depression_data.csv"
file_path_2 = "dataset/OxCGRT_simplified_v1.csv"
url_path_1 = "https://data.cdc.gov/api/views/8pt5-q6wp/rows.csv?accessType=DOWNLOAD"
url_path_2 = "https://github.com/OxCGRT/covid-policy-dataset.git"

if not os.path.exists(file_path_1):
    print("Downloading CDC depression dataset from data.gov ...")
    urllib.request.urlretrieve(
        url_path_1,
        file_path_1
    )
else:
    print("Dataset already exists. Skipping download.")

if not os.path.exists(file_path_2):
    print("Downloading OxCGRT dataset from GitHub ...")
    # Clone the repository and move the file
    os.system("git clone " + url_path_2)
    os.system("mv covid-policy-dataset/data/OxCGRT_simplified_v1.csv dataset/")
    os.system("rm -rf covid-policy-dataset")
else:
    print("Dataset already exists. Skipping download.")


Dataset already exists. Skipping download.
Dataset already exists. Skipping download.


### 1.2 Load the Datasets using Pandas

In [66]:
# Load the datasets
import pandas as pd
import numpy as np

depression_data = pd.read_csv(file_path_1)
oxcgrt_data = pd.read_csv(file_path_2)

# Display the column names of each dataset
print("CDC Depression Dataset Columns:")
print(depression_data.columns)
print("\nOxCGRT Dataset Columns:")
print(oxcgrt_data.columns)

CDC Depression Dataset Columns:
Index(['Indicator', 'Group', 'State', 'Subgroup', 'Phase', 'Time Period',
       'Time Period Label', 'Time Period Start Date', 'Time Period End Date',
       'Value', 'Low CI', 'High CI', 'Confidence Interval', 'Quartile Range'],
      dtype='str')

OxCGRT Dataset Columns:
Index(['CountryName', 'CountryCode', 'RegionName', 'RegionCode',
       'Jurisdiction', 'Date', 'C1M_combined_numeric', 'C1M_combined',
       'C2M_combined_numeric', 'C2M_combined', 'C3M_combined_numeric',
       'C3M_combined', 'C4M_combined_numeric', 'C4M_combined',
       'C5M_combined_numeric', 'C5M_combined', 'C6M_combined_numeric',
       'C6M_combined', 'C7M_combined_numeric', 'C7M_combined',
       'C8EV_combined_numeric', 'C8EV_combined', 'E1_combined_numeric',
       'E1_combined', 'E2_combined_numeric', 'E2_combined',
       'H1_combined_numeric', 'H1_combined', 'H2_combined_numeric',
       'H2_combined', 'H3_combined_numeric', 'H3_combined',
       'H6M_combined_numeric'

/tmp/ipykernel_74239/659960727.py:6: DtypeWarning: Columns (0: RegionName, 1: RegionCode, 2: MajorityVaccinated, 3: PopulationVaccinated) have mixed types. Specify dtype option on import or set low_memory=False.
  oxcgrt_data = pd.read_csv(file_path_2)


### 1.3 - Filter our datasets (Clean up)

In [67]:
# Filter the simplified OxCGRT dataset for the United States
oxcgrt_us = oxcgrt_data[oxcgrt_data['CountryName'] == 'United States']

# Compare the size of the dataset before and after filtering
print("\nSize of the original OxCGRT dataset:", oxcgrt_data.shape)
print("Size of the filtered OxCGRT dataset (United States):", oxcgrt_us.shape)

# Remove some unnecessary columns that we won't be using for our analysis
columns_to_drop = ['CountryName', 'CountryCode', 'RegionCode', 'Jurisdiction']
oxcgrt_us = oxcgrt_us.drop(columns=columns_to_drop)

# Our dataset contains two versions of the measurements: *_combined and *_combined_numeric.
# For this assignment we will only keep the *_combined_numeric columns, as they are easier to work with for analysis and visualization.
# We might want to include the *_combined columns in a future assignment when we do more detailed analysis
columns_to_drop = [col for col in oxcgrt_us.columns if col.endswith('_combined') and not col.endswith('_combined_numeric')]
oxcgrt_us = oxcgrt_us.drop(columns=columns_to_drop)

oxcgrt_us.to_csv("dataset/OxCGRT_simplified_v2.csv", index=False)


Size of the original OxCGRT dataset: (390909, 50)
Size of the filtered OxCGRT dataset (United States): (56992, 50)


### 1.4 Make sure the dataframes are comparable by date and state

In [ ]:
# Fill missing values in the 'RegionName' column with 'United States' (as they are country-level data)
oxcgrt_us.loc[oxcgrt_us['RegionName'].isna() | (oxcgrt_us['RegionName'] == ''), 'RegionName'] = 'United States'

# Get unique time periods and state combinations from the depression dataset
unique_time_state_combs = depression_data[['Time Period Start Date', 'Time Period End Date', 'State']].drop_duplicates()

print("\nUnique time period and state combinations in the depression dataset:")
print(unique_time_state_combs)

# Convert 'Time Period Start Date' and 'Time Period End Date' to int YYYYMMDD format
unique_time_state_combs['Time Period Start Date'] = pd.to_datetime(
    unique_time_state_combs['Time Period Start Date'], format='%m/%d/%Y'
).dt.strftime('%Y%m%d').astype('int64')
unique_time_state_combs['Time Period End Date'] = pd.to_datetime(
    unique_time_state_combs['Time Period End Date'], format='%m/%d/%Y', errors='coerce'
).dt.strftime('%Y%m%d').astype('int64')

print("\nUnique time period and state combinations with formatted dates:")
print(unique_time_state_combs)

# Sort the rows of the OxCGRT dataset into groups, based on the unique time-state combinations from the depression dataset
# Add 'Start' and 'End' columns to the OxCGRT dataset and fill them based on the unique time-state combinations
# These will be matched to the depression dataset later when we do the merge, so we can easily filter OxCGRT data for the relevant time periods and states
oxcgrt_us['Start'] = np.nan
oxcgrt_us['End'] = np.nan

for start, end, state in unique_time_state_combs.values:
    mask = (
        (oxcgrt_us['RegionName'] == state) &
        (oxcgrt_us['Date'] >= start) &
        (oxcgrt_us['Date'] <= end)
    )
    oxcgrt_us.loc[mask, 'Start'] = start
    oxcgrt_us.loc[mask, 'End'] = end

# Now we can drop the 'Date' column, as we have the 'Start' and 'End' columns to indicate the relevant time periods for each row
oxcgrt_us = oxcgrt_us.drop(columns=['Date'])

oxcgrt_us.to_csv("dataset/OxCGRT_simplified_v4.csv", index=False)

# Convert the 'PopulationVaccinated' column to numeric
# (It was imported as an object, need to convert it to numeric for aggregation)
oxcgrt_us['PopulationVaccinated'] = pd.to_numeric(oxcgrt_us['PopulationVaccinated'])

# Aggregate the OxCGRT data by group (unique time-state combinations) using median for numeric columns
oxcgrt_aggregated = (
    oxcgrt_us
    .dropna(subset=['Start', 'End'])
    .groupby(['RegionName', 'Start', 'End'], as_index=False)
    .median(numeric_only=True)
)

# Add the 'MajorityVaccinated' column back to the aggregated dataset
oxcgrt_aggregated['MajorityVaccinated'] = (
    oxcgrt_us
    .groupby(['RegionName', 'Start', 'End'])['MajorityVaccinated']
    .first()
    .values
)

# Save the aggregated dataset to a new CSV file
oxcgrt_aggregated.to_csv("dataset/OxCGRT_simplified_v5.csv", index=False)



Unique time period and state combinations in the depression dataset:
      Time Period Start Date Time Period End Date          State
0                 04/23/2020           05/05/2020  United States
19                04/23/2020           05/05/2020        Alabama
20                04/23/2020           05/05/2020         Alaska
21                04/23/2020           05/05/2020        Arizona
22                04/23/2020           05/05/2020       Arkansas
...                      ...                  ...            ...
16633             08/20/2024           09/16/2024       Virginia
16634             08/20/2024           09/16/2024     Washington
16635             08/20/2024           09/16/2024  West Virginia
16636             08/20/2024           09/16/2024      Wisconsin
16637             08/20/2024           09/16/2024        Wyoming

[3754 rows x 3 columns]

Unique time period and state combinations with formatted dates:
       Time Period Start Date  Time Period End Date         

In [78]:
# Now we can merge the depression dataset with the aggregated OxCGRT dataset based on the unique time-state combinations
# Beforehand we need to convert the 'Time Period Start Date' and 'Time Period End Date' columns in the depression dataset
converted_depression_data = depression_data.copy()
converted_depression_data['Time Period Start Date'] = pd.to_datetime(
    converted_depression_data['Time Period Start Date'], format='%m/%d/%Y'
).dt.strftime('%Y%m%d').astype('int64')
converted_depression_data['Time Period End Date'] = pd.to_datetime(
    converted_depression_data['Time Period End Date'], format='%m/%d/%Y', errors='coerce'
).dt.strftime('%Y%m%d').astype('int64')

merged = converted_depression_data.merge(
    oxcgrt_aggregated,
    left_on=['State', 'Time Period Start Date', 'Time Period End Date'],
    right_on=['RegionName', 'Start', 'End'],
    how='left'
)

# Clean up merged dataset
merged = merged.drop(columns=['Time Period Start Date', 'Time Period End Date', 'RegionName', 'Start', 'End'])

# Save the merged dataset to a new CSV file
merged.to_csv("dataset/merged_dataset.csv", index=False)

In [79]:
merged.shape
merged.info()
merged.describe()

# - df.shape
# - df.info()
# - df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 16794 entries, 0 to 16793
Data columns (total 40 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Indicator                        16794 non-null  str    
 1   Group                            16794 non-null  str    
 2   State                            16794 non-null  str    
 3   Subgroup                         16794 non-null  str    
 4   Phase                            16794 non-null  str    
 5   Time Period                      16794 non-null  int64  
 6   Time Period Label                16794 non-null  str    
 7   Value                            16087 non-null  float64
 8   Low CI                           16087 non-null  float64
 9   High CI                          16087 non-null  float64
 10  Confidence Interval              16087 non-null  str    
 11  Quartile Range                   11017 non-null  str    
 12  C1M_combined_numeric         

,Time Period,Value,Low CI,High CI,C1M_combined_numeric,C2M_combined_numeric,C3M_combined_numeric,C4M_combined_numeric,C5M_combined_numeric,C6M_combined_numeric,...,V2..summary.,V3..summary.,V4..summary.,ConfirmedCases,ConfirmedDeaths,PopulationVaccinated,StringencyIndex_Average,GovernmentResponseIndex_Average,ContainmentHealthIndex_Average,EconomicSupportIndex
count,16794.000000,16087.000000,16087.000000,16087.000000,11877.000000,11877.000000,11877.000000,11877.000000,11877.000000,11877.000000,...,11877.000000,11877.000000,11877.000000,1.187700e+04,1.187700e+04,11877.000000,11877.000000,11877.000000,11877.000000,11877.000000
mean,35.922830,28.140946,24.642270,31.893181,1.463627,0.969816,0.894923,1.945946,0.371306,0.681990,...,1.601162,3.225057,0.241728,1.492034e+07,2.090730e+05,29.924560,48.297837,52.989226,54.462056,42.687547
std,21.530312,8.951691,8.593666,9.481899,0.912992,0.777111,0.685421,1.685966,0.446951,0.578677,...,1.343676,2.386730,0.428148,2.786538e+07,3.411859e+05,29.187824,18.467618,11.804268,10.632237,26.705726
min,1.000000,4.600000,3.300000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,3.600000e+02,7.000000e+00,0.000000,17.330000,28.030000,32.040000,0.000000
25%,17.000000,22.100000,18.700000,25.500000,0.500000,0.500000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,1.307050e+05,2.308000e+03,0.000000,29.860000,41.550000,44.660000,25.000000
50%,37.000000,27.700000,24.100000,31.600000,1.500000,1.000000,1.000000,2.500000,0.500000,0.500000,...,2.000000,5.000000,0.000000,8.320660e+05,1.309100e+04,29.510000,51.390000,54.200000,55.940000,37.500000
75%,55.000000,33.400000,29.700000,37.400000,2.500000,1.500000,1.500000,3.500000,0.500000,1.000000,...,3.000000,5.000000,0.000000,1.144430e+07,2.495230e+05,58.350000,64.810000,63.020000,63.720000,62.500000
max,72.000000,85.200000,79.900000,89.500000,3.000000,3.000000,2.000000,4.000000,2.000000,2.000000,...,3.000000,5.000000,1.000000,9.972925e+07,1.086179e+06,87.220000,93.520000,80.210000,80.830000,100.000000


**Part 2: Required Exploratory Questions**

You must answer at least four specific EDA questions using visualization.

Each question must:

- Be clearly stated
- Use Pandas operations
- Include one iplot() visualization
- Include a written interpretation of 3 to 5 full sentences

You may not submit screenshots without explanation.

---

**Required Visualization Types**

You must include the following:

**1. Category Counts (Bar Chart)**

Example structure:

- df[‘category’].value_counts().iplot(kind=‘bar’)

Question type example:

- Which category appears most frequently?

---

**2. Grouped Aggregation (Bar Chart)**

Example structure:

- df.groupby(‘group’)[‘numeric’].mean().iplot(kind=‘bar’)

Question type example:

- Which group has the highest average value?

---

**3. Distribution (Histogram)**

Example structure:

- df[‘numeric’].iplot(kind=‘hist’)

Question type example:

- Is the distribution symmetric or skewed?
- Are there potential outliers?

---

**4. Trend or Top-N Comparison**

If time or ordered data exists:

- df.groupby(‘year’)[‘numeric’].mean().iplot(kind=‘line’)

OR
- df.sort_values(‘numeric’, ascending=False).head(10).iplot(kind=‘bar’)

Question type example:

- How does the main variable change over time?
- What are the top 10 highest values?


**Part 3: Required Interpretation**

For each visualization, answer the following in full sentences:

1. What question are you asking?
2. What method did you use?
3. What does the visualization show?
4. What insight can you draw from it?

Minimum 3 sentences per visualization.  
Clarity and reasoning matter more than aesthetics.

---

**Optional - for future reports**

- Identifying rows lost during joins
- Using Boolean filtering or query()
- Identifying missing data patterns


**Grading Criteria**

Your Phase 2 submission will be evaluated on:

- Correct implementation of join (25 points)
- Dataset meets size and structure requirements (15 points)
- Quality and clarity of EDA questions (20 points)
- Correct use of Pandas operations (15 points)
- Appropriate use of required visualizations (15 points)
- Depth and clarity of written interpretation (10 points)

**Total: 100 points**

---

This phase is about learning how to think with data.  
Plot → Observe → Interpret.  
Strong exploratory analysis will make modeling significantly easier in later phases.